# Аналитическая записка

## Задача

Продолжаем историю, что вы являетесь аналитиком и руководителем сервиса Прайм))  

Ваша задача придумать бизнес-гипотезу и проверить ее с помощью данных. Также дополнительно надо будет понять что с этим всем делать

Для того, чтобы целиком все знать для решения проекта надо подождать пока мы пройдем 5-6 недели лекции и семинара, но много было в лекции 3-4 + будет на семинарах. Настоятельно рекомендую вам начать пораньше, тут может быть много работы 


## Что сделать

1. **Сформулировать бизнес-вопрос.** Проверяемый и осмысленный для бизнеса. «Посмотреть на данные» - не вопрос. «Почему в сентябре упала выручка от подписок и что с этим делать» - вопрос. (Можно уже начать думать)
2. **Рассчитать релевантные метрики** и обосновать зерно расчета: что такое одна строка вашей таблицы - клиент, подписка, платеж, канал × день - и почему для вашего вопроса нужно именно такое. За это обоснование - второй критерий в таблице ниже. (В целом метрики какие-то уже тоже можно посчитать)
3. **Выполнить факторную декомпозицию** изменения ключевого показателя за период. (Это вот как раз лекция 5-6)
4. **Сделать 3–5 графиков**, каждый из которых что-то доказывает. (Графики вам поможет сделать клодик, советую seaborn, чтобы было красиво и интерактивно)
5. **Написать вывод и рекомендацию.**

## Выгрузка данных

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

In [ ]:
CLIENT_FILTER = q("select client_filter from prime.assignments where login = current_user").iloc[0, 0]

print(CLIENT_FILTER)

In [ ]:
WINDOW = ("2026-01-01", "2026-07-01")   # окно фактов: первое полугодие 2026, правый конец не включен
SEGMENT = f"select client_id from prime.clients where {CLIENT_FILTER}"

clients = q(f"""
    select client_id, registered_at, region_code, acquisition_channel
    from prime.clients
    where {CLIENT_FILTER}
    order by client_id
""")

subs = q(f"""
    select subscription_id, owner_client_id, tariff_id, started_at, ended_at, is_trial, cancel_reason
    from prime.subscriptions
    where owner_client_id in ({SEGMENT})
    order by subscription_id
""")

pay = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status, p.payment_method
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where s.owner_client_id in ({SEGMENT})
      and p.paid_at >= :a and p.paid_at < :b
    order by p.payment_id
""", a=WINDOW[0], b=WINDOW[1])

sub_members = q(f"""
    select m.subscription_id, m.client_id as member_id
    from prime.subscription_members m
    join prime.subscriptions s on s.subscription_id = m.subscription_id
    where s.owner_client_id in ({SEGMENT})
    order by m.subscription_id, m.client_id
""")

members = q(f"""
    select subscription_id, client_id
    from prime.subscription_members
    where client_id in ({SEGMENT})
    order by client_id, subscription_id
""")

tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

spend = q("select channel, spend_date, amount from prime.marketing_spend order by channel, spend_date")

registrations = q("""
    select acquisition_channel as channel, registered_at::date as reg_date, count(*) as n_clients
    from prime.clients
    group by 1, 2
    order by 1, 2
""")

N = len(clients)
print("ЗЕРНО витрины: одна строка - один клиент сегмента. Клиентов:", N)
print()
for name, frame in [("clients", clients), ("subs", subs), ("pay", pay), ("sub_members", sub_members),
                    ("members", members), ("tariffs", tariffs), ("spend", spend),
                    ("registrations", registrations)]:
    print(f"{name:<14} {len(frame):>9} строк")

In [ ]:
table_df = [clients, subs, pay, sub_members, members, tariffs, spend, registrations]
 
for i in table_df:
    display(i.head(1))

## 1. Бизнес-вопрос
Так как для бизнеса одним из самых важных финансовых показателей является прибыль, а выручка это одна из частей прибыли, рассмотрим вопрос изменения выручки ПРАЙМА. Для этого порассуждаем над тем за счет чего меняются поступления от подписок и на что нашему бизнесу стоит обратить внимание относительно этой сферы. Посмотрим на то какие данные у нас есть. 

Прежде чем обсуждать выручку, хотим еще раз убедиться, что одни и те же записи не учтены повторно и что у платежей известны сумма и дата

In [ ]:
pay["paid_at"] = pd.to_datetime(pay["paid_at"], utc=True)

quality = pd.DataFrame({
    "Данные": ["clients", "subs", "pay"],
    "Повторы номера": [clients["client_id"].duplicated().sum(),
                       subs["subscription_id"].duplicated().sum(),
                       pay["payment_id"].duplicated().sum()],
    "Записи без номера": [clients["client_id"].isna().sum(),
                          subs["subscription_id"].isna().sum(),
                          pay["payment_id"].isna().sum()],
})

print("Платежей без суммы:", pay["amount"].isna().sum())
print("Платежей с нулевой или отрицательной суммой:", (pay["amount"] <= 0).sum())
print("Платежей без даты:", pay["paid_at"].isna().sum())
quality

В исходной выгрузке ключи заполнены и не повторяются, суммы платежей положительные и это хорошо! Конечно, бывают и случаи когда с человека случайно списали деньги дважды или более раз, поэтому будем еще проверять такой случай, при котором списали одинаковую сумму с небольшой разницей по времени

### Какие месяцы стоит сравнить?
Хотим выбрать периоды, для которых сравнение выручки будет осмысленным. Если за отдельные дни нет записей или один месяц заметно короче другого,
это может повлиять на итог и создать впечатление роста или падения бизнеса. Рассмотрим какие периоды доступны и есть ли дни без платежей.

In [ ]:
days = pd.date_range(WINDOW[0], WINDOW[1], freq="D", inclusive="left")
payments_by_day = pay.groupby(pay["paid_at"].dt.date).size()
payments_by_day = payments_by_day.reindex(days.date, fill_value=0)

print("Первый платеж:", pay["paid_at"].min())
print("Последний платеж:", pay["paid_at"].max())
print("Дни без платежей:")

COMPARE_MONTHS = ["2026-01", "2026-03"]
payments_by_day[payments_by_day == 0]

Есть платежи за январь–июнь 2026 года. За 14 и 15 апреля записей нет. Непонятно действительно ли в эти дни не было платежей или они не попали в выгрузку, поэтому апрель не берем в сравнение.

Выбираем **январь и март**. В обоих месяцах по 31 дню, и за каждый день есть записи. Разница в продолжительности месяцев не будет объяснять изменение выручки. Однако надо понимать, что сезонность и особенности поведения клиентов при этом могут оставаться.

### Стали ли клиенты платить больше за подписку?
Для ответа на этот вопрос посчитаем сумму успешного списания (`success`) в январе и марте внутри каждого тарифа. Это поможет понять, есть ли какие-то основания связывать изменение выручки с более высоким платежом за подписку. Неуспешные списания и возвраты не включаем в выручку по принятому в курсе определению. Сопоставляем фактически уплаченные суммы, так как именно они показывают, какие деньги получил сервис. Также убедимся, что все успешные платежи учтены при разделении поступлений по тарифам

In [ ]:
paid = pay[pay["status"] == "success"].copy()
paid["month"] = paid["paid_at"].dt.strftime("%Y-%m")

paid_tariffs = paid.merge(
    subs[["subscription_id", "tariff_id"]],
    on="subscription_id", how="left",
)

print("Успешных платежей до добавления тарифа:", len(paid))
print("После добавления тарифа:", len(paid_tariffs))
print("Платежей без указанного тарифа:", paid_tariffs["tariff_id"].isna().sum())

paid_tariffs["tariff"] = paid_tariffs["tariff_id"].map({1: "Базовый", 2: "Расширенный"})
price_check = paid_tariffs[paid_tariffs["month"].isin(COMPARE_MONTHS)]
price_check = price_check.groupby(["month", "tariff"])["amount"].agg(["min", "max"])
price_check.columns = ["Минимальный платеж, ₽", "Максимальный платеж, ₽"]
price_check

In [ ]:
plot_data = paid_tariffs[paid_tariffs["month"].isin(COMPARE_MONTHS)].copy()
plot_data["month"] = plot_data["month"].map({
    "2026-01": "Январь 2026", "2026-03": "Март 2026"
})

sns.set_theme(style="whitegrid", font_scale=1.05)
fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=plot_data, x="tariff", y="amount", hue="month",
    order=["Базовый", "Расширенный"],
    hue_order=["Январь 2026", "Март 2026"],
    palette=["#CBD0DA", "#6D28D9"],
    width=0.65, gap=0.1, errorbar=None, ax=ax
)

for bars in ax.containers:
    ax.bar_label(bars, fmt="%.0f ₽", padding=5, fontsize=11)

ax.set_title(
    "В марте средний платеж выше по обоим тарифам",
    loc="left", pad=18, fontweight="bold"
)
ax.set_xlabel("Тариф подписки")
ax.set_ylabel("Средний успешный платеж, ₽")
ax.set_ylim(0, plot_data["amount"].max() * 1.25)
ax.legend(title=None, frameon=False, ncol=2, loc="upper left")
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.show()

Количество платежей сохранилось. В марте сумма успешного списания выше январской и по Базовому, и по Расширенному тарифу. Внутри каждого тарифа за месяц суммы одинаковые.

Значит, у роста среднего платежа есть основание помимо изменения популярности дорогого тарифа. Однако более высокий платеж сам по себе не гарантирует рост общей выручки. А важно нам, сколько подписок продолжает приносить деньги и сколько успешных списаний приходится на каждую из них.

### **Бизнес-вопрос:** за счет чего изменилась выручка от подписок в нашем сегменте в марте по сравнению с январем в 2026 году?

## 2. Метрики
Наш сегмент состоит из клиентов, совершавших покупки через приложение в июне 2025 года. Рассматриваем поступления от принадлежащих им подписок в 2026 году. 

Для сравнения соберем таблицу, в которой одна строка - один месяц, один тариф и один канал
привлечения владельца подписки. Подписку считаем платящей, если за месяц по ней прошел хотя бы один успешный платеж. Владелец и подключенные им близкие не становятся несколькими отдельными плательщиками одной подписки.


Все показатели считаем за один месяц, по успешным платежам после исключения предполагаемых повторных списаний.

`Выручка` = сумма успешных платежей

`Платящие подписки` = количество уникальных подписок с успешным платежом за месяц

`Успешные платежи` = количество успешных списаний без предполагаемых повторов

`Средний платеж` = выручка / количество успешных платежей

`Платежей на платящую подписку` = количество успешных платежей / количество платящих подписок

Проверим, не списывали ли с одной подписки одинаковую сумму несколько раз за короткое время. Проверяем только успешные
платежи, чтобы не принять обычную повторную попытку после отказа за двойное списание

In [ ]:
ordered_paid = paid.sort_values(["subscription_id", "amount", "paid_at"]).copy()
ordered_paid["amount_kopecks"] = (ordered_paid["amount"] * 100).round().astype("int64")
gap = ordered_paid.groupby(["subscription_id", "amount"])["paid_at"].diff()

repeat_check = []
for i in [5, 10, 15, 20, 60]:
    repeat_check.append({
        "Интервал, минут": i,
        "Предполагаемые повторные списания": (gap < pd.Timedelta(minutes=i)).sum(),
    })

pd.DataFrame(repeat_check)

При границах от 15 до 60 минут получается 197 предполагаемых повторных списаний. Для анализа примем, что если по одной подписке повторно прошла та же сумма менее чем через 20 минут, оставляем первое
списание

In [ ]:
is_repeat = gap < pd.Timedelta(minutes=20)
clean_paid = ordered_paid[~is_repeat].copy()

raw_revenue = ordered_paid.groupby("month")["amount_kopecks"].sum() / 100
clean_revenue = clean_paid.groupby("month")["amount_kopecks"].sum() / 100

cleaning_effect = pd.DataFrame({
    "Поступления по выгрузке, ₽": raw_revenue,
    "Выручка для анализа, ₽": clean_revenue,
}).loc[COMPARE_MONTHS]
cleaning_effect["Разница, ₽"] = raw_revenue - clean_revenue
cleaning_effect["Разница, %"] = (raw_revenue - clean_revenue) / raw_revenue * 100
cleaning_effect.round(3)

В январе исключение предполагаемых повторов уменьшает сумму на 7571, в марте на 14107. Всего в этих двух месяцах исключаем 72 повторных платежа. Дальше под выручкой для анализа понимаем сумму без этих предполагаемых повторов

Определим, какие тарифы и каналы привлечения стоят за успешными платежами января и марта. Канал относится к владельцу подписки. Платежи клиентов с неизвестным каналом сохраним в отдельной группе без канала. Также проверим, что при добавлении сведений о владельцах не появились лишние платежи и не потерялись суммы.

In [ ]:
facts = clean_paid[clean_paid["month"].isin(COMPARE_MONTHS)].copy()
rows_before = len(facts)
revenue_before = facts["amount_kopecks"].sum() / 100

facts = facts.merge(
    subs[["subscription_id", "owner_client_id", "tariff_id"]],
    on="subscription_id", how="left",
)
facts = facts.merge(
    clients[["client_id", "acquisition_channel"]],
    left_on="owner_client_id", right_on="client_id", how="left",
)
facts["channel"] = facts["acquisition_channel"].fillna("без канала")
facts["tariff"] = facts["tariff_id"].map({1: "Базовый", 2: "Расширенный"})

print("Платежей без найденного владельца:", facts["client_id"].isna().sum())
print("Платежей без найденного тарифа:", facts["tariff"].isna().sum())
pd.DataFrame({
    "Этап": ["До добавления сведений", "После добавления сведений"],
    "Платежей": [rows_before, len(facts)],
    "Сумма, ₽": [revenue_before, facts["amount_kopecks"].sum() / 100],
})

По итогу сохранились все 69241 платеж выбранных месяцев и вся их сумма. Для каждого платежа определены владелец и тариф. Таким образом,
разбивка по группам не создает дополнительных поступлений. У 1461 платежа канал привлечения владельца не указан. Они остаются
в анализе в группе без канала и входят в общую выручку.

Для каждого сочетания месяца, тарифа и канала посчитаем выручку, число успешных платежей и число платящих подписок. Например, одна строка покажет поступления от Базового тарифа в канале organic за январь. Одну подписку внутри месяца относим к одному тарифу и одному каналу
ее владельца

In [ ]:
monthly = facts.groupby(["month", "tariff", "channel"], as_index=False).agg(
    revenue_kopecks=("amount_kopecks", "sum"),
    n_payments=("payment_id", "size"),
    paying_subs=("subscription_id", "nunique"),
)
monthly["revenue"] = monthly["revenue_kopecks"] / 100

print("Строк в расчетной таблице:", len(monthly))
monthly[["month", "tariff", "channel", "paying_subs", "n_payments", "revenue"]].head()

Получилось 28 строк, из них два месяца, два тарифа и семь групп
каналов, включая без канала. Теперь уже посмотрим на общую картину
по каждому месяцу.

In [ ]:
metrics = monthly.groupby("month")[["revenue_kopecks", "n_payments", "paying_subs"]].sum()
metrics = metrics.loc[COMPARE_MONTHS]
metrics["revenue"] = metrics["revenue_kopecks"] / 100
metrics["avg_payment"] = metrics["revenue"] / metrics["n_payments"]
metrics["payments_per_sub"] = metrics["n_payments"] / metrics["paying_subs"]

metric_names = {
    "revenue": "Выручка для анализа, ₽",
    "paying_subs": "Платящие подписки",
    "n_payments": "Успешные платежи",
    "avg_payment": "Средний платеж, ₽",
    "payments_per_sub": "Платежей на платящую подписку",
}
metrics[list(metric_names)].rename(columns=metric_names).round({
    "Выручка для анализа, ₽": 2,
    "Средний платеж, ₽": 2,
    "Платежей на платящую подписку": 4,
})

**Выручка выросла с 9 908 208 ₽ до 10 812 001 ₽**, хотя число платящих подписок **сократилось с 34 953
до 32 303**. При этом средний успешный платеж стал выше, а число платежей на одну платящую подписку осталось близким к прежнему уровню.

In [ ]:
revenue_change = metrics.loc["2026-03", "revenue"] - metrics.loc["2026-01", "revenue"]
print("Изменение выручки, ₽:", revenue_change)

change = (metrics.loc["2026-03", list(metric_names)] / metrics.loc["2026-01", list(metric_names)] - 1) * 100
change = change.rename(index=metric_names)
change.round(2).to_frame("Изменение, %")

Сокращение числа платящих подписок еще не означает, что все эти клиенты ушли из сервиса. Здесь мы видим только наличие успешной оплаты в конкретном месяце

Убедимся, что выручка, собранная по тарифам и каналам, совпадает с суммой
платежей выбранных месяцев. Такая сверка нужна, чтобы дальнейшее объяснение
роста опиралось на весь денежный результат, а не на случайно оставшуюся часть.

In [ ]:
reconciliation = pd.DataFrame({
    "По платежам, ₽": facts.groupby("month")["amount_kopecks"].sum() / 100,
    "По расчетной таблице, ₽": metrics["revenue_kopecks"] / 100,
})
reconciliation["Расхождение, ₽"] = reconciliation["По расчетной таблице, ₽"] - reconciliation["По платежам, ₽"]
reconciliation

Суммы по группам полностью совпадают с суммами подготовленных платежей. 

## 3. Факторная декомпозиция

Сначала определим, какие тарифы и каналы дали прирост. Затем разделим его на изменение числа платящих подписок, частоты оплат и среднего платежа. Для всех расчетов сохраняем январь и март 2026 года, границы месяцев по UTC и правило исключения предполагаемых повторных списаний.

Хотим отличить рост, который наблюдается во всем сегменте, от результата одного удачного тарифа или канала. В каждой разбивке должны сохраниться все 903 793 ₽ прироста. Разбивки по тарифам и каналам показывают одни и те же деньги с разных сторон: складывать их между собой нельзя.

In [ ]:
by_tariff = monthly.pivot_table(
    index="tariff", columns="month", values="revenue", aggfunc="sum", fill_value=0
)
by_channel = monthly.pivot_table(
    index="channel", columns="month", values="revenue", aggfunc="sum", fill_value=0
)

for name, table in [("Тарифы", by_tariff), ("Каналы", by_channel)]:
    table["Изменение, ₽"] = table["2026-03"] - table["2026-01"]
    table["Изменение, %"] = (table["2026-03"] / table["2026-01"] - 1) * 100
    display(table.rename(columns={"2026-01": "Январь, ₽", "2026-03": "Март, ₽"}).round(2))
    print("Расхождение с общим приростом, ₽:", round(table["Изменение, ₽"].sum() - revenue_change, 2))

**Базовый тариф дал 695 079 ₽, или 76,9% общего прироста**, Расширенный 208 714 ₽. Выручка выросла в каждом канале, но неравномерно: от 0,8% в social
до 13,8% в группе без канала. Крупнейший вклад organic 365 507 ₽


`Выручка = платящие подписки * платежей на платящую подписку * средний платеж`

Сначала оценим изменение числа платящих подписок при январской частоте и сумме платежа. Затем учтем новую
частоту оплат, а последним средний платеж. Совместное влияние изменений относится к фактору, который
меняем позже. Здесь база означает только подписки с успешной оплатой, опять же

In [ ]:
jan = metrics.loc["2026-01"]
mar = metrics.loc["2026-03"]

effects = pd.Series({
    "Платящие подписки": (mar["paying_subs"] - jan["paying_subs"])
        * jan["payments_per_sub"] * jan["avg_payment"],
    "Частота оплат": mar["paying_subs"]
        * (mar["payments_per_sub"] - jan["payments_per_sub"]) * jan["avg_payment"],
    "Средний платеж": mar["n_payments"] * (mar["avg_payment"] - jan["avg_payment"]),
})

print("Сумма влияний, ₽:", round(effects.sum(), 2))
print("Фактический прирост, ₽:", revenue_change)
print("Расхождение, ₽:", round(abs(effects.sum() - revenue_change), 2))
effects.round(2).to_frame("Вклад в изменение выручки, ₽")

Сокращение платящей базы дало **−751 202 ₽**, частота оплат **−3 917 ₽**, средний платеж **1 658 911 ₽**. Сумма влияний совпала с приростом выручки. Частота почти не изменилась. Тогда чем объясняется
рост среднего платежа? Средний платеж может вырасти, даже если цены остались прежними, достаточно, чтобы дорогой тариф стал занимать большую долю оплат
Оценим мартовский набор платежей по январским ценам каждого тарифа. Разница среднего платежа с январем покажет влияние структуры. Затем заменим
цены на мартовские. Цены берем из фактических оплат.

`Средний платеж = сумма по тарифам (доля тарифа в платежах × цена его платежа)`

In [ ]:
tariff_metrics = monthly.groupby(["month", "tariff"])[
    ["revenue", "n_payments", "paying_subs"]
].sum()
tariff_metrics["price"] = tariff_metrics["revenue"] / tariff_metrics["n_payments"]
prices = tariff_metrics["price"].unstack("month")
payment_counts = tariff_metrics["n_payments"].unstack("month")
shares = payment_counts / payment_counts.sum()
march_at_old_prices = (shares["2026-03"] * prices["2026-01"]).sum()

detailed_effects = effects.drop("Средний платеж").copy()
detailed_effects["Доли тарифов"] = mar["n_payments"] * (march_at_old_prices - jan["avg_payment"])
detailed_effects["Цена внутри тарифов"] = mar["n_payments"] * (mar["avg_payment"] - march_at_old_prices)

constant_price_revenue = (payment_counts["2026-03"] * prices["2026-01"]).sum()
print("Выручка марта при январских ценах, ₽:", round(constant_price_revenue))
print("Изменение при январских ценах, %:", round((constant_price_revenue / jan["revenue"] - 1) * 100, 2))
print("Доля Расширенного тарифа в платежах, %:", (shares.loc["Расширенный"] * 100).round(2).to_dict())
print("Расхождение четырех вкладов с приростом, ₽:", round(abs(detailed_effects.sum() - revenue_change), 2))
detailed_effects.round(2).to_frame("Вклад в изменение выручки, ₽")

**Повышение цены внутри тарифов дало 1 662 450 ₽.** Доля Расширенного тарифа в платежах почти не изменилась 38,14% на 38,09%. Ее небольшой сдвиг дал −3 539 ₽. Таким образом, рост среднего платежа практически полностью
связан с более высокой суммой списания внутри тарифов.

При январских ценах тот же мартовский набор оплат принес бы 9 149 551 ₽, на 7,66% меньше января. Это расчет при фиксированном фактическом числе платежей, а не прогноз того, как повели бы себя клиенты без повышения цен. Покажем вклады в рублях. Каждый следующий шаг добавляется к предыдущему, а последний столбец
показывает итоговый прирост.

In [ ]:
steps = detailed_effects / 1000
labels = ["Платящая\nбаза", "Частота\nоплат", "Доли\nтарифов", "Цена внутри\nтарифов"]
before = steps.cumsum() - steps

sns.set_theme(style="whitegrid", font_scale=1.05)
fig, ax = plt.subplots(figsize=(10, 5.4))
bars = ax.bar(labels, steps, bottom=before, width=0.62, color=["#CBD0DA", "#CBD0DA", "#CBD0DA", "#6D28D9"], zorder=3)
total = ax.bar("Итог", steps.sum(), width=0.62, color="#4B5563", zorder=3)
for i in range(len(steps) - 1):
    ax.plot([i + 0.31, i + 0.69], [steps.cumsum().iloc[i]] * 2, color="#9CA3AF", lw=1)
ax.bar_label(bars, labels=[f"{x:+,.0f}".replace(",", " ") for x in steps], padding=6)
ax.bar_label(total, labels=[f"{steps.sum():+,.0f}".replace(",", " ")], padding=6)
ax.axhline(0, color="#374151", linewidth=1)
ax.set_title("Цена дала +1,66 млн ₽; сокращение базы поглотило 0,75 млн ₽",
             loc="left", pad=18, fontweight="bold", fontsize=13)
ax.set_ylabel("Вклад в изменение выручки, тыс. ₽")
ax.set_xlabel("Январь -> март 2026 · порядок подстановок слева направо", labelpad=12)
ax.set_ylim(-1050, 1200)
ax.grid(axis="x", visible=False)
sns.despine(left=True, bottom=True)
fig.text(0.01, 0.01, "Успешные платежи сегмента, без предполагаемых повторов; месяцы по UTC.", fontsize=9, color="#6B7280")
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

Рост выручки сопровождается сокращением числа плательщиков. Поэтому одного показателя поступлений недостаточно для оценки состояния подписки. Проверим также обратный порядок трех основных факторов. Он распределит
совместный эффект иначе, но не должен менять общий результат

In [ ]:
reverse_effects = pd.Series({
    "Платящие подписки": (mar["paying_subs"] - jan["paying_subs"])
        * mar["payments_per_sub"] * mar["avg_payment"],
    "Частота оплат": jan["paying_subs"]
        * (mar["payments_per_sub"] - jan["payments_per_sub"]) * mar["avg_payment"],
    "Средний платеж": jan["n_payments"] * (mar["avg_payment"] - jan["avg_payment"]),
})
order_check = pd.DataFrame({
    "База -> частота -> платеж, ₽": effects,
    "Платеж -> частота -> база, ₽": reverse_effects,
})
print("Расхождение обратного порядка с приростом, ₽:", round(abs(reverse_effects.sum() - revenue_change), 2))
order_check.round(2)

При обратном порядке вклад среднего платежа выше, а потери от сокращения базы больше по модулю. Средний платеж компенсирует уменьшение базы; частота оплат почти не влияет. Для дальнейших выводов используем первоначальный порядок, указанный под каскадом.

## 4. Где искать проблему и насколько устойчив рост

### Выручка может скрывать сокращение плательщиков

Сравним изменение выручки и платящей базы внутри каждого канала относительно его собственного января. Это позволит увидеть риск и в небольшом канале, который трудно заметить на фоне крупных сумм organic.

In [ ]:
channel_base = monthly.pivot_table(
    index="channel", columns="month", values="paying_subs", aggfunc="sum", fill_value=0
)
channel_change = pd.DataFrame({
    "Платящие подписки": (channel_base["2026-03"] / channel_base["2026-01"] - 1) * 100,
    "Выручка": by_channel["Изменение, %"],
}).sort_values("Платящие подписки")
channel_plot = channel_change.reset_index().melt(
    id_vars="channel", var_name="Показатель", value_name="Изменение, %"
)

fig, ax = plt.subplots(figsize=(10, 6.2))
sns.barplot(data=channel_plot, y="channel", x="Изменение, %", hue="Показатель",
            order=channel_change.index, hue_order=["Платящие подписки", "Выручка"],
            palette=["#6D28D9", "#CBD0DA"], errorbar=None, ax=ax)
for bars in ax.containers:
    ax.bar_label(bars, fmt="%+.1f%%", padding=4, fontsize=10)
ax.axvline(0, color="#374151", linewidth=1)
ax.set_title("В social платящих подписок меньше на 14,2%, а выручка почти не растет",
             loc="left", pad=55, fontweight="bold", fontsize=12)
ax.set(xlabel="Изменение к январю, % · январь -> март 2026", ylabel="Канал привлечения владельца",
       xlim=(-19, 18))
ax.legend(title=None, frameon=False, ncol=2, loc="lower right", bbox_to_anchor=(1, 1.01), fontsize=10)
ax.grid(axis="y", visible=False)
sns.despine(left=True, bottom=True)
fig.text(0.01, 0.01, "Источник: успешные платежи сегмента без предполагаемых повторов; база — уникальные платящие подписки.", fontsize=9, color="#6B7280")
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

**Платящая база сократилась во всех семи группах**, хотя выручка в каждой выросла. В social снижение самое сильное: **3 755 -> 3 220 подписок**, а поступления прибавили всего **8 309 ₽ (+0,8%)**.

По абсолютному сокращению также выделяются performance (−582 подписки) и organic (−529). Поэтому social — приоритет для проверки доли потерянных плательщиков, но работа только с ним не решит проблему всего сегмента.
Это сравнение поведения существующей группы клиентов, а не оценка окупаемости рекламных каналов: расходы на привлечение здесь не сопоставлялись.

### Кто перестал платить и кто появился? 

Теперь одна строка, одна подписка с оплатой хотя бы в одном из двух месяцев. Разделим их на три непересекающиеся группы: платили в обоих месяцах, только в январе или только в марте. 

Только в марте означает отсутствие январской оплаты, а не обязательно новую подписку: среди таких плательщиков могут быть и вернувшиеся.

In [ ]:
subscription_months = facts.pivot_table(
    index="subscription_id", columns="month", values="amount_kopecks",
    aggfunc="sum", fill_value=0
) / 100
subscription_months["Группа"] = "Платили в обоих месяцах"
subscription_months.loc[subscription_months["2026-03"] == 0, "Группа"] = "Платили только в январе"
subscription_months.loc[subscription_months["2026-01"] == 0, "Группа"] = "Платили только в марте"

movement = subscription_months.groupby("Группа").agg(
    Подписок=("2026-01", "size"), Январь=("2026-01", "sum"), Март=("2026-03", "sum")
)
movement["Изменение выручки, ₽"] = movement["Март"] - movement["Январь"]
print("Расхождение вкладов групп с приростом, ₽:", round(movement["Изменение выручки, ₽"].sum() - revenue_change, 2))
movement.rename(columns={"Январь": "Январь, ₽", "Март": "Март, ₽"})

Из 34 953 январских плательщиков **30 907 платили и в марте**, а **4 046 нет**.
Взамен появилось **1 396** подписок с мартовской оплатой без январской: это
компенсировало лишь **34,5%** исчезнувших из платящей базы. Поэтому итоговая
база уменьшилась на 2 650 подписок.

Платившие в обоих месяцах дали +1 575 702 ₽; группа без мартовской оплаты
−1 134 281 ₽; появившиеся в марте +462 372 ₽. Это еще один способ разложить
тот же прирост, поэтому эти суммы не добавляем к вкладам цены и базы.


Проследим за 4 046 подписками без мартовской оплаты до конца июня. Увидим, по скольким из них снова появились успешные платежи. Этот взгляд из будущих месяцев нужен для диагностики; он не участвует в объяснении мартовской выручки.

In [ ]:
missing_march = subscription_months.index[
    subscription_months["Группа"] == "Платили только в январе"
]
later_payments = clean_paid[clean_paid["month"].isin(["2026-04", "2026-05", "2026-06"])]
returned_ids = later_payments.loc[
    later_payments["subscription_id"].isin(missing_march), "subscription_id"
].unique()
followup = pd.DataFrame({
    "Результат": ["Успешные оплаты\nснова появились", "Успешных оплат\n в апреле–июне нет"],
    "Подписок": [len(returned_ids), len(missing_march) - len(returned_ids)],
})

fig, ax = plt.subplots(figsize=(9, 4.8))
sns.barplot(data=followup, x="Результат", y="Подписок", hue="Результат",
            palette=["#6D28D9", "#CBD0DA"], legend=False, errorbar=None, width=0.55, ax=ax)
for bars in ax.containers:
    ax.bar_label(bars, labels=[f"{b.get_height():,.0f}       {b.get_height()/len(missing_march):.0%}".replace(",", " ")
                              for b in bars], padding=7, fontsize=12, fontweight="bold")
ax.set_title("У 34,3% подписок без мартовской оплаты платежи появились позже",
             loc="left", pad=18, fontweight="bold", fontsize=12)
ax.set(xlabel="4 046 подписок: платили в январе, не платили в марте", ylabel="Количество подписок",
       ylim=(0, 3300))
ax.grid(axis="x", visible=False)
sns.despine(left=True, bottom=True)
fig.text(0.01, 0.01, "апрель–июнь 2026, UTC. За 14–15 апреля в выгрузке нет записей.", fontsize=9, color="#6B7280")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

**По 1 389 подпискам (34,3%) оплаты появились снова.** Это доля возобновивших
оплату среди январских плательщиков без мартовской оплаты, а не показатель
удержания всех клиентов. По остальным 2 657 подпискам оплат в доступной
выгрузке апреля–июня нет; это также не доказывает окончательный уход.

Проверим, что известно о статусе подписок к началу апреля и были ли по
продолжающимся подпискам неуспешные или возвращенные платежи в марте.
Так мы сможем отделить работу с отменами от проверки оплаты.

In [ ]:
missing_details = subs[subs["subscription_id"].isin(missing_march)].copy()
missing_details["ended_at"] = pd.to_datetime(missing_details["ended_at"], utc=True)
ended_before_april = missing_details["ended_at"] < pd.Timestamp("2026-04-01", tz="UTC")
ongoing_ids = missing_details.loc[~ended_before_april, "subscription_id"]
unsuccessful_march = pay[
    (pay["paid_at"].dt.strftime("%Y-%m") == "2026-03")
    & pay["status"].isin(["failed", "refunded"])
]

print("Закончились до апреля:", ended_before_april.sum())
print("Не закончились до апреля:", len(ongoing_ids))
print("Из них с failed/refunded в марте:", ongoing_ids.isin(unsuccessful_march["subscription_id"]).sum())
print("Из них с успешной оплатой в апреле–июне:", ongoing_ids.isin(returned_ids).sum())
print("Оплаты позже при дате окончания до апреля:",
      missing_details.loc[ended_before_april, "subscription_id"].isin(returned_ids).sum())

У **2 653** подписок указано окончание до апреля. Еще **1 393** к этому моменту
не закончились, у всех есть мартовская запись `failed` или `refunded`,
а **1 387** впоследствии снова успешно оплатили подписку. Это основание
отдельно проверить неуспешные списания и возвраты, а не отправлять всем
одинаковую скидку для «ушедших».

По двум подпискам успешный платеж появился позже указанной даты окончания.
Это несогласованность статуса и платежей, которую стоит уточнить у владельца
данных. График выше описывает факт оплаты, не факт восстановления договора.
Пропуск 14–15 апреля может занижать число наблюдаемых возобновлений.

### Сколько сокращения базы выдержит выручка

Оценим запас относительно январского уровня выручки. Предположим, что в
следующем сопоставимом 31-дневном периоде цена и число платежей на платящую
подписку останутся мартовскими, а база сократится на заданную долю.

**Выручка сценария = мартовская выручка × (1 − доля сокращения платящей базы).**

**Порог сокращения = 1 − январская выручка / мартовская выручка.**

Рассмотрим каждый тариф и сегмент в целом. Для общей строки дополнительно
сохраняем мартовское соотношение тарифов. Это условная модель чувствительности:
она показывает последствия выбранного сценария, но не предсказывает отток
и не оценивает реакцию клиентов на цену.

In [ ]:
scenario_base = by_tariff[["2026-01", "2026-03"]].copy()
scenario_base.loc["Весь сегмент"] = [jan["revenue"], mar["revenue"]]
loss_limit = (1 - scenario_base["2026-01"] / scenario_base["2026-03"]) * 100

scenarios = []
for segment, row in scenario_base.iterrows():
    for loss in np.arange(0, 15.1, 0.5):
        scenarios.append({
            "Сегмент": segment,
            "Сокращение базы, %": loss,
            "Изменение к январю, %": (row["2026-03"] * (1 - loss / 100) / row["2026-01"] - 1) * 100,
        })
scenarios = pd.DataFrame(scenarios)
loss_limit.round(2).to_frame("Дополнительное сокращение базы до январской выручки, %")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.7))
sns.lineplot(data=scenarios, x="Сокращение базы, %", y="Изменение к январю, %",
             hue="Сегмент", style="Сегмент", errorbar=None,
             palette={"Базовый": "#A8ADB7", "Расширенный": "#6D28D9", "Весь сегмент": "#4B5563"},
             linewidth=2.4, ax=ax)
ax.axhline(0, color="#374151", linewidth=1)
for segment in loss_limit.index:
    limit = loss_limit[segment]
    color = "#6D28D9" if segment == "Расширенный" else "#6B7280"
    ax.scatter(limit, 0, color=color, s=45, zorder=4)
    ax.annotate(f"{limit:.1f}%", (limit, 0), xytext=(0, 8), textcoords="offset points",
                ha="center", color=color, fontweight="bold")
ax.set_title("Расширенный тариф потеряет прирост при сокращении базы еще на 3,7%",
             loc="left", pad=20, fontweight="bold", fontsize=12)
ax.set(xlabel="Дополнительное сокращение платящей базы относительно марта, %",
       ylabel="Выручка относительно января 2026, %", xlim=(0, 15), ylim=(-15, 18))
ax.legend(title=None, frameon=False, loc="upper right")
sns.despine()
fig.text(0.01, 0.01, "Условие: 31 день, мартовские цены и частота оплат; для всего сегмента доли тарифов неизменны.", fontsize=9, color="#6B7280")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

У Расширенного тарифа запас относительно января небольшой. Сокращение платящей базы еще на 3,67% вернет выручку Расширенного тарифа к январскому уровню при сохранении мартовских цен и частоты оплат. У Базового тарифа порог **13,56%**, у сегмента в целом — **8,36%**. На графике пороги обозначены
точками пересечения с нулевым изменением выручки

## 5. Вывод и рекомендация

**Что случилось.** Выручка от подписок выбранного сегмента в марте 2026 года составила **10 812 001 ₽**, на **903 793 ₽ (+9,12%)** больше января. При этом платящих подписок стало меньше на **2 650 (−7,58%)**. Январь и март содержат по 31 дню, а различие продолжительности месяцев не объясняет результат.

**За счет чего.** При порядке база -> частота -> доли тарифов -> цена повышение цены дало **+1 662 450 ₽**, сокращение базы **−751 202 ₽**, частота оплат **−3 917 ₽**, изменение долей тарифов **−3 539 ₽**. Точная сумма вкладов равна приросту, отображаемые целые рубли округлены. При январских ценах мартовские оплаты дали бы снижение выручки на **7,66%**. Больше денег поступило благодаря более высокой цене, а не расширению платящей базы.

**Чего мы не знаем, а хотелось бы** Анализ описывает клиентов, покупавших через приложение в июне 2025 года, а не весь сервис. Сравнение двух месяцев и факторное разложение не доказывают, что именно цена вызвала сокращение базы, возможны сезонность, отмены, отказы оплаты и другие причины. Мы также не оцениваем прибыль и окупаемость каналов. Два расхождения между датой окончания и поздней оплатой, а также отсутствие записей за 14–15 апреля ограничивают выводы о жизненном цикле подписки.

**Что делать.** В первую неделю после анализа руководителю подписки вместе с командой платежей проверить 1 393 продолжавшиеся подписки без мартовской успешной оплаты. Разделить отказы и возвраты, сопоставить их с последующими списаниями. Две оплаты после указанного окончания передать системным аналитикам на сверку. По 2 653 закончившимся подпискам CRM-команде отдельно разобрать
причины отмен,  ретроспективный список не использовать для рассылки без обновления текущего статуса.

В регулярном отчете показывать выручку рядом с числом платящих подписок, отдельно для social и Расширенного тарифа. Для social проверить причины сокращения базы и сравнить их с другими каналами; его почти нулевой рост выручки уже сопровождается потерей 14,2% плательщиков. Для Расширенного тарифа контролировать небольшой запас роста — 3,67% по сценарной модели.

Если проверка подтвердит проблему оплаты, протестировать напоминание или обновление платежного способа на случайно выбранной части подходящих подписок с контрольной группой. Через 31 день сравнить долю подписок
с успешной оплатой и выручку на подписку во всей назначенной группе, включая нулевые оплаты. Дополнительно проверить возвраты и повторные списания. Масштабировать меру по ее дополнительному результату и стоимости, а не по общему росту выручки после повышения цены